<a href="https://colab.research.google.com/github/briskicedteaa/DopaGen/blob/main/DopaGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython
from Bio import SeqIO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.0 MB/s  0:00:00


In [ ]:
import zipfile
import os

zip_paths = [
    "/content/D1Receptor.zip",
    "/content/D2Receptor.zip",
    "/content/D3Receptor.zip",
    "/content/D4Receptor.zip",
    "/content/D5Receptor.zip"
]

extract_folder = "/content/PDB"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [ ]:
zip_paths1 = [
    "/content/COMT.zip",
    "/content/DAT.zip",
    "/content/DDC.zip",
    "/content/MAOB.zip",
    "/content/VMAT.zip"
]

extract_folder = "/content/PDB1"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths1:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [ ]:
zip_paths2 = [
    "/content/CatecholO.zip",
    "/content/DA-Transpoer.zip",
    "/content/MOA-A.zip"
]

extract_folder = "/content/PDB2"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths2:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [ ]:
zip_paths3 = [
    "/content/5-HT2A.zip",
    "/content/AdenosineA2A.zip",
    "/content/NTSR1.zip",
    "/content/GPCR88.zip"
]

extract_folder = "/content/PDB3"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths3:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [ ]:
import pandas as pd

files = [
    "/content/PDB/D1.csv",
    "/content/PDB/D2.csv",
    "/content/PDB/D3.csv",
    "/content/PDB/D4.csv",
    "/content/PDB/D5.csv",
    "/content/PDB1/COMT.csv",
    "/content/PDB1/DAT.csv",
    "/content/PDB1/DDC.csv",
    "/content/PDB1/MAOB.csv",
    "/content/PDB2/CatecholO.csv",
    "/content/PDB2/DA-Transporter.csv",
    "/content/PDB2/MOA-A.csv",
    "/content/PDB3/5-HT2A.csv",
    "/content/PDB3/AdenosineA2A.csv",
    "/content/PDB3/NTSR1.csv",
    "/content/PDB3/GPCR88.csv"
]

df = pd.concat([pd.read_csv(f, sep=";") for f in files], ignore_index=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/PDB/D1.csv'

In [ ]:
features = [
    "Molecule ChEMBL ID",
    "Smiles",
    "Target ChEMBL ID",
    "Action Type",
]

df = df[features].copy()
df.head()

In [ ]:
df = df[features].dropna().copy()

df = df.drop_duplicates(subset=["Molecule ChEMBL ID"]).copy()

print(df["Action Type"].value_counts())
print("Remaining Ligands:", len(df))
df.groupby("Molecule ChEMBL ID")["Action Type"].nunique().value_counts()
df.groupby("Molecule ChEMBL ID").nunique().value_counts()

In [ ]:
df.groupby(["Target ChEMBL ID", "Action Type"]).size().unstack(fill_value=0)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/ligands_dataset.csv")

df = df[
    ~df["Action Type"].str.contains(
        "ACTIVATOR",
        case=False,
        na=False
    )
].copy()

df = df[
    ~df["Action Type"].str.contains(
        "INVERSE",
        case=False,
        na=False
    )
].copy()

df = df[
    ~df["Action Type"].str.contains(
        "DISRUPTING",
        case=False,
        na=False
    )
].copy()

df = df[
    ~df["Action Type"].str.contains(
        "SUBSTRATE",
        case=False,
        na=False
    )
].copy()

df = df[df["Action Type"] != "MODULATOR"].copy()

print(df["Action Type"].value_counts())
print("Remaining Ligands:", len(df))
df.groupby("Molecule ChEMBL ID")["Action Type"].nunique().value_counts()
df.groupby("Molecule ChEMBL ID").nunique().value_counts()

Action Type
INHIBITOR                        2072
ANTAGONIST                       1199
AGONIST                           816
BINDING AGENT                     769
PARTIAL AGONIST                   135
POSITIVE ALLOSTERIC MODULATOR      57
NEGATIVE ALLOSTERIC MODULATOR      46
Name: count, dtype: int64
Remaining Ligands: 5094


,,,count
Smiles,Target ChEMBL ID,Action Type,
1,1,1,5094


In [ ]:
df.to_csv('/content/ligands_dataset.csv', index=False)

In [3]:
import sys
!{sys.executable} -m pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 58.1 MB/s  0:00:00


In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd

df = pd.read_csv("/content/ligands_dataset.csv")

def get_morgan_fp(smiles, radius=2, n_bits=2048):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            arr = np.zeros((1,), dtype=np.int8)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            return arr
    except:
        pass
    return None

df['Morgan_Fingerprint'] = df['Smiles'].apply(get_morgan_fp)
display(df[['Molecule ChEMBL ID', 'Smiles', 'Morgan_Fingerprint']].head())

,Molecule ChEMBL ID,Smiles,Morgan_Fingerprint
0,CHEMBL5179930,O=C(O)C(=O)O.O=C1CCc2ccccc2N1CCCN1CCN(c2ccc(Cl...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CHEMBL5169993,O=C(O)C(=O)O.c1ccc(N2CCN(CCCN3CCCc4ccccc43)CC2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CHEMBL5196870,O=C(O)C(=O)O.O=C1CCc2ccccc2N1CCCN1CCN(c2ccc(Cl...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CHEMBL267014,Clc1ccc(N2CCN(Cc3c[nH]c4ncccc34)CC2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CHEMBL5172201,O=C(O)C(=O)O.O=C1CCc2ccccc2N1CCCN1CCN(c2ccccc2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [6]:
from sklearn.model_selection import GroupShuffleSplit

X = np.array(df["Morgan_Fingerprint"].tolist())

y = df["Target ChEMBL ID"].values

groups = df["Molecule ChEMBL ID"].values

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4116
Testing samples: 1030


In [7]:
from sklearn.ensemble import RandomForestClassifier

target_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

target_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [8]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = target_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.883495145631068

Classification Report:
               precision    recall  f1-score   support

   CHEMBL1843       1.00      1.00      1.00         1
   CHEMBL1850       0.60      0.60      0.60         5
   CHEMBL1951       0.81      0.43      0.57        30
   CHEMBL2023       1.00      1.00      1.00         7
   CHEMBL2039       0.84      0.99      0.91       313
   CHEMBL2056       0.95      0.59      0.73        32
    CHEMBL217       0.86      0.90      0.88        92
    CHEMBL219       0.83      0.74      0.78       100
    CHEMBL224       0.94      0.92      0.93       126
    CHEMBL234       0.82      0.78      0.80        74
    CHEMBL238       0.92      0.46      0.62        26
    CHEMBL251       0.97      0.95      0.96       191
CHEMBL3399910       1.00      1.00      1.00        27
   CHEMBL4123       1.00      1.00      1.00         6

     accuracy                           0.88      1030
    macro avg       0.90      0.81      0.84      1030
 weighted a

In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=target_model.classes_
)

cm_df = pd.DataFrame(
    cm,
    index=target_model.classes_,
    columns=target_model.classes_
)

print(cm_df)

               CHEMBL1843  CHEMBL1850  CHEMBL1951  CHEMBL2023  CHEMBL2039  \
CHEMBL1843              1           0           0           0           0   
CHEMBL1850              0           3           0           0           0   
CHEMBL1951              0           0          13           0          17   
CHEMBL2023              0           0           0           7           0   
CHEMBL2039              0           0           3           0         309   
CHEMBL2056              0           0           0           0           8   
CHEMBL217               0           0           0           0           3   
CHEMBL219               0           0           0           0           9   
CHEMBL224               0           1           0           0           3   
CHEMBL234               0           1           0           0           1   
CHEMBL238               0           0           0           0          10   
CHEMBL251               0           0           0           0           7   

In [10]:
target_encoded = pd.get_dummies(df["Target ChEMBL ID"])

X_action = np.hstack([X, target_encoded.to_numpy()])

y_action = df["Action Type"].values

X_action_train = X_action[train_idx]
X_action_test = X_action[test_idx]

y_action_train = y_action[train_idx]
y_action_test = y_action[test_idx]

action_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

action_model.fit(X_action_train, y_action_train)

print("Model trained.")

Model trained.


In [11]:
y_action_pred = action_model.predict(X_action_test)

print("Accuracy:", accuracy_score(y_action_test, y_action_pred))
print(classification_report(y_action_test, y_action_pred, zero_division=0))

Accuracy: 0.9223300970873787
                               precision    recall  f1-score   support

                    ACTIVATOR       0.00      0.00      0.00         0
                      AGONIST       0.92      0.93      0.93       148
                   ANTAGONIST       0.91      0.90      0.91       242
                BINDING AGENT       0.89      0.89      0.89       158
             DISRUPTING AGENT       0.00      0.00      0.00         2
                    INHIBITOR       0.94      0.97      0.96       414
              INVERSE AGONIST       0.80      0.44      0.57         9
                    MODULATOR       1.00      1.00      1.00         2
NEGATIVE ALLOSTERIC MODULATOR       1.00      0.92      0.96        13
              PARTIAL AGONIST       0.96      0.74      0.84        31
POSITIVE ALLOSTERIC MODULATOR       0.90      0.82      0.86        11

                     accuracy                           0.92      1030
                    macro avg       0.76      

In [28]:
def predict_smiles(smiles, target_id=None):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError("Invalid SMILES string.")

    fp = generator.GetFingerprint(mol)
    fingerprint = np.zeros(2048, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fingerprint)

    target_probs = target_model.predict_proba(
        fingerprint.reshape(1, -1)
    )[0]

    target_scores = sorted(
        zip(target_model.classes_, target_probs),
        key=lambda x: x[1],
        reverse=True
    )

    if target_id is None:
        target_id = target_scores[0][0]

    if target_id not in target_encoded.columns:
        raise ValueError("Target ID not found in training data.")

    target_vector = np.array([
        int(col == target_id)
        for col in target_encoded.columns
    ])

    action_features = np.concatenate([
        fingerprint, target_vector
    ]).reshape(1, -1)

    action_probs = action_model.predict_proba(action_features)[0]

    action_scores = sorted(
        zip(action_model.classes_, action_probs),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "target_predictions": target_scores,
        "selected_target": target_id,
        "action_predictions": action_scores
    }

In [29]:
train_df = df.iloc[train_idx].copy()
train_df.to_csv("dopamine_train_set.csv", index=False)

test_df = df.iloc[test_idx].copy()
test_df.to_csv("dopamine_test_set.csv", index=False)

In [31]:
results = predict_smiles(
    test_df["Smiles"].iloc[0],
    target_id="CHEMBL219"
)

print("TARGET PREDICTIONS")

for target, score in results["target_predictions"]:
    print(f"{target}: {score * 100:.2f}%")

print("\nACTION TYPE PREDICTIONS")

for action, score in results["action_predictions"]:
    print(f"{action}: {score * 100:.2f}%")

TARGET PREDICTIONS
CHEMBL219: 97.00%
CHEMBL251: 1.33%
CHEMBL217: 1.00%
CHEMBL2039: 0.33%
CHEMBL2056: 0.33%
CHEMBL1843: 0.00%
CHEMBL1850: 0.00%
CHEMBL1951: 0.00%
CHEMBL2023: 0.00%
CHEMBL224: 0.00%
CHEMBL234: 0.00%
CHEMBL238: 0.00%
CHEMBL3399910: 0.00%
CHEMBL4123: 0.00%

ACTION TYPE PREDICTIONS
ANTAGONIST: 87.67%
BINDING AGENT: 10.33%
PARTIAL AGONIST: 1.00%
INHIBITOR: 0.67%
AGONIST: 0.33%
ACTIVATOR: 0.00%
DISRUPTING AGENT: 0.00%
INVERSE AGONIST: 0.00%
MODULATOR: 0.00%
NEGATIVE ALLOSTERIC MODULATOR: 0.00%
POSITIVE ALLOSTERIC MODULATOR: 0.00%
SUBSTRATE: 0.00%
